In [ ]:
-- =============================================================================
-- SLEEPER TRADES PIPELINE
-- Dynasty-Aware Multi-Season Trade Analysis
-- 
-- Key Features:
-- - Multi-season player point tracking
-- - Draft pick career point tracking
-- - Trade completeness detection (all picks realized)
-- - Dynasty vs redraft league support
-- - Trade impact calculation across multiple time horizons
-- =============================================================================

In [ ]:
-- ---------- STAGING ----------

In [ ]:
-- Base trade transactions with league context
CREATE OR REPLACE MATERIALIZED VIEW stg_trade_transactions AS
SELECT
  t.league_id,
  li.season,
  t.transaction_id,
  t.status,
  t.creator,
  t.adds,
  t.drops,
  t.roster_ids,
  t.draft_picks,
  t.leg AS week,
  t.created,
  to_timestamp(t.created/1000.0) AS event_ts,
  lower(regexp_replace(coalesce(li.name,'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key,
  li.name AS cluster_name
FROM workspace.sleeper_raw.sleeper_transactions_snapshot t
JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id)
WHERE t.type = 'trade' AND t.status = 'complete';

In [ ]:
-- Trade metadata with league type
CREATE OR REPLACE MATERIALIZED VIEW dim_trade_metadata AS
SELECT
  t.league_id,
  t.transaction_id,
  t.season AS trade_season,
  t.week AS trade_week,
  t.event_ts AS trade_date,
  t.cluster_key,
  t.cluster_name,
  lc.league_type,
  lc.season_start AS league_start_season,
  lc.season_end AS league_end_season
FROM stg_trade_transactions t
LEFT JOIN workspace.sleeper_core.dim_league_config lc ON t.league_id = lc.league_id;

In [ ]:
-- ---------- TRADE COMPLETENESS ----------

In [ ]:
-- Determine if all picks in a trade have been realized
CREATE OR REPLACE MATERIALIZED VIEW dim_trade_completeness AS
WITH pick_status AS (
  SELECT
    league_id,
    transaction_id,
    COUNT(*) AS total_picks,
    SUM(CASE WHEN is_realized THEN 1 ELSE 0 END) AS realized_picks,
    MAX(pick_season) AS latest_pick_season
  FROM bridge_trade_pick_to_player
  GROUP BY league_id, transaction_id
),
trade_metadata AS (
  SELECT
    transaction_id,
    league_id,
    season AS trade_season
  FROM stg_trade_transactions
)
SELECT
  tm.transaction_id,
  tm.league_id,
  tm.trade_season,
  COALESCE(ps.total_picks, 0) AS total_picks,
  COALESCE(ps.realized_picks, 0) AS realized_picks,
  ps.latest_pick_season,
  -- Trade is complete if it has no picks OR all picks are realized
  CASE
    WHEN ps.total_picks IS NULL THEN TRUE
    WHEN ps.total_picks = ps.realized_picks THEN TRUE
    ELSE FALSE
  END AS is_complete
FROM trade_metadata tm
LEFT JOIN pick_status ps USING (league_id, transaction_id);